# Pazarlama Kampanyası Yanıt Tahmini Lojistik Regresyon Modeli

Bu çalışmada pazarlama kampanyası veri seti kullanılarak, bir müşterinin en son kampanyayı kabul edip etmeyeceğini (Response) tahmin eden bir lojistik regresyon modeli sıfırdan kurulmuştur. Veri temizliği, özellik mühendisliği, sınıf dengesizliği analizi, model eğitimi, eşik değeri optimizasyonu ve hiperparametre araması dahil olmak üzere sürecin tüm adımları ayrıntılı grafiklerle açıklanmıştır.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, precision_recall_curve,
    confusion_matrix, classification_report
)

import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (10, 6)
sns.set_style('whitegrid')

In [ ]:
df = pd.read_csv('/content/marketing_campaign.csv', sep='\t')
df.head()

Response: Müşterinin en son pazarlama kampanyasını kabul edip etmediği (1 = Kabul etti, 0 = Etmedi). Bu çalışmadaki hedef değişkendir.

Income: Yıllık hane geliri.

Year_Birth: Doğum yılı.

Education: Eğitim seviyesi.

Marital_Status: Medeni durum.

Kidhome / Teenhome: Evdeki küçük çocuk ve genç sayısı.

Mnt* sütunları: Farklı ürün kategorilerinde yapılan harcama tutarları.

Num*Purchases sütunları: Farklı kanallardan yapılan satın alma sayıları.

Recency: Son alışverişten bu yana geçen gün sayısı.

NumWebVisitsMonth: Aylık web sitesi ziyaret sayısı.

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

In [ ]:
df.duplicated().sum()

In [ ]:
df['Response'].value_counts()

In [ ]:
oranlar = df['Response'].value_counts(normalize=True) * 100

plt.figure(figsize=(7, 5))
sns.barplot(x=oranlar.index.map({0: 'Kabul Etmedi', 1: 'Kabul Etti'}), y=oranlar.values, palette='Set2')
plt.ylabel('Yüzde (%)')
plt.title('Hedef Değişken (Response) Dağılımı')
for i, deger in enumerate(oranlar.values):
    plt.text(i, deger + 1, f'%{deger:.1f}', ha='center')
plt.tight_layout()
plt.show()

Hedef değişkenin dengesiz dağıldığı görülmektedir; kampanyayı kabul eden müşteriler azınlıktadır. Bu durum modelin değerlendirilmesinde yalnızca doğruluk (accuracy) yerine kesinlik (precision), duyarlılık (recall), F1 skoru ve ROC-AUC gibi metriklerin de kullanılmasını gerektirmektedir.

In [ ]:
copy_df = df.copy()
copy_df = copy_df.dropna()
print(f'Eksik değer temizliği sonrası satır sayısı: {len(copy_df)}')

In [ ]:
copy_df['Education'].unique()

In [ ]:
egitim_haritasi = {
    'Basic': 0,
    'Graduation': 1,
    '2n Cycle': 2,
    'PhD': 3,
    'Master': 4
}
copy_df['Education'] = copy_df['Education'].map(egitim_haritasi)

In [ ]:
copy_df['Marital_Status'].unique()

In [ ]:
birliktelik_haritasi = {
    'Single': 0,
    'Together': 1,
    'Married': 2,
    'Divorced': 3,
    'Widow': 4,
    'Alone': 1,
    'Absurd': 1,
    'YOLO': 1
}
copy_df['Marital_Status'] = copy_df['Marital_Status'].map(birliktelik_haritasi)

## Özellik Mühendisliği

Modelin performansını artırmak için doğum yılından yaş, harcama sütunlarının toplamından toplam harcama, çocuk sütunlarının toplamından toplam çocuk sayısı ve satın alma kanallarının toplamından toplam satın alma sayısı türetilmiştir.

In [ ]:
copy_df['Age'] = 2024 - copy_df['Year_Birth']

harcama_sutunlari = ['MntWines', 'MntFruits', 'MntMeatProducts', 'MntFishProducts', 'MntSweetProducts', 'MntGoldProds']
copy_df['Total_Spending'] = copy_df[harcama_sutunlari].sum(axis=1)

copy_df['Total_Children'] = copy_df['Kidhome'] + copy_df['Teenhome']

satin_alma_sutunlari = ['NumDealsPurchases', 'NumWebPurchases', 'NumCatalogPurchases', 'NumStorePurchases']
copy_df['Total_Purchases'] = copy_df[satin_alma_sutunlari].sum(axis=1)

copy_df[['Age', 'Total_Spending', 'Total_Children', 'Total_Purchases']].describe()

## Keşifsel Veri Analizi

Bu bölümde kampanyayı kabul eden ve etmeyen müşteri gruplarının özellik bazında nasıl farklılaştığı incelenmiştir.

In [ ]:
onemli_ozellikler = ['Income', 'Age', 'Total_Spending', 'Total_Children', 'Total_Purchases', 'Recency', 'NumWebVisitsMonth']

fig, axes = plt.subplots(3, 3, figsize=(16, 13))
axes = axes.flatten()

for i, ozellik in enumerate(onemli_ozellikler):
    sns.boxplot(data=copy_df, x='Response', y=ozellik, ax=axes[i], palette='Set3')
    axes[i].set_xticklabels(['Kabul Etmedi', 'Kabul Etti'])
    axes[i].set_title(f'Response Bazında {ozellik}')

for j in range(len(onemli_ozellikler), len(axes)):
    axes[j].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(12, 10))
korelasyon_sutunlari = onemli_ozellikler + ['Education', 'Marital_Status', 'Response']
sns.heatmap(copy_df[korelasyon_sutunlari].corr(), annot=True, fmt='.2f', cmap='coolwarm', square=True)
plt.title('Özellikler ve Hedef Değişken Arası Korelasyon Matrisi')
plt.tight_layout()
plt.show()

In [ ]:
egitim_yaniti = copy_df.groupby('Education')['Response'].mean() * 100

plt.figure(figsize=(8, 5))
sns.barplot(x=egitim_yaniti.index, y=egitim_yaniti.values, palette='crest')
plt.xlabel('Eğitim Seviyesi (Kodlanmış)')
plt.ylabel('Kampanyayı Kabul Etme Oranı (%)')
plt.title('Eğitim Seviyesine Göre Kampanya Kabul Oranı')
plt.tight_layout()
plt.show()

## Modelleme İçin Veri Hazırlığı

In [ ]:
ozellikler = ['Income', 'Age', 'Education', 'Marital_Status', 'Total_Children', 'Total_Spending',
              'Total_Purchases', 'Recency', 'NumWebVisitsMonth']

X = copy_df[ozellikler]
y = copy_df['Response']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f'Eğitim seti boyutu: {X_train.shape[0]}')
print(f'Test seti boyutu: {X_test.shape[0]}')
print(f'Eğitim setinde pozitif sınıf oranı: %{y_train.mean() * 100:.2f}')
print(f'Test setinde pozitif sınıf oranı: %{y_test.mean() * 100:.2f}')

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## Temel Lojistik Regresyon Modeli

Öncelikle sınıf ağırlıklandırması yapılmadan bir temel model kurulmuştur.

In [ ]:
temel_model = LogisticRegression(max_iter=500, random_state=42)
temel_model.fit(X_train_scaled, y_train)

temel_tahmin = temel_model.predict(X_test_scaled)
temel_olasilik = temel_model.predict_proba(X_test_scaled)[:, 1]

In [ ]:
temel_metrikler = pd.DataFrame({
    'Metrik': ['Doğruluk', 'Kesinlik', 'Duyarlılık', 'F1 Skoru', 'ROC-AUC'],
    'Değer': [
        accuracy_score(y_test, temel_tahmin),
        precision_score(y_test, temel_tahmin),
        recall_score(y_test, temel_tahmin),
        f1_score(y_test, temel_tahmin),
        roc_auc_score(y_test, temel_olasilik)
    ]
})
temel_metrikler

In [ ]:
cm = confusion_matrix(y_test, temel_tahmin)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Kabul Etmedi', 'Kabul Etti'], yticklabels=['Kabul Etmedi', 'Kabul Etti'])
plt.xlabel('Tahmin Edilen')
plt.ylabel('Gerçek')
plt.title('Temel Model Karmaşıklık Matrisi')
plt.tight_layout()
plt.show()

In [ ]:
print(classification_report(y_test, temel_tahmin, target_names=['Kabul Etmedi', 'Kabul Etti']))

## ROC Eğrisi ve Kesinlik-Duyarlılık Eğrisi

Sınıf dengesizliği bulunan problemlerde ROC eğrisinin yanı sıra kesinlik-duyarlılık eğrisi de modelin performansını değerlendirmek için önemlidir.

In [ ]:
fpr, tpr, esik_roc = roc_curve(y_test, temel_olasilik)
roc_auc = roc_auc_score(y_test, temel_olasilik)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='#2980b9', linewidth=2, label=f'ROC Eğrisi (AUC = {roc_auc:.3f})')
plt.plot([0, 1], [0, 1], 'k--', linewidth=1)
plt.xlabel('Yanlış Pozitif Oranı')
plt.ylabel('Doğru Pozitif Oranı')
plt.title('ROC Eğrisi')
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

In [ ]:
precision_degerleri, recall_degerleri, esik_pr = precision_recall_curve(y_test, temel_olasilik)

plt.figure(figsize=(8, 6))
plt.plot(recall_degerleri, precision_degerleri, color='#c0392b', linewidth=2)
plt.xlabel('Duyarlılık (Recall)')
plt.ylabel('Kesinlik (Precision)')
plt.title('Kesinlik-Duyarlılık Eğrisi')
plt.tight_layout()
plt.show()

## Sınıf Dengesizliği İçin Ağırlıklandırılmış Model

Azınlık sınıfının (kampanyayı kabul edenler) daha iyi yakalanabilmesi için class_weight='balanced' parametresiyle model yeniden eğitilmiştir.

In [ ]:
dengeli_model = LogisticRegression(max_iter=500, class_weight='balanced', random_state=42)
dengeli_model.fit(X_train_scaled, y_train)

dengeli_tahmin = dengeli_model.predict(X_test_scaled)
dengeli_olasilik = dengeli_model.predict_proba(X_test_scaled)[:, 1]

In [ ]:
karsilastirma = pd.DataFrame({
    'Metrik': ['Doğruluk', 'Kesinlik', 'Duyarlılık', 'F1 Skoru', 'ROC-AUC'],
    'Temel Model': [
        accuracy_score(y_test, temel_tahmin),
        precision_score(y_test, temel_tahmin),
        recall_score(y_test, temel_tahmin),
        f1_score(y_test, temel_tahmin),
        roc_auc_score(y_test, temel_olasilik)
    ],
    'Dengeli Model': [
        accuracy_score(y_test, dengeli_tahmin),
        precision_score(y_test, dengeli_tahmin),
        recall_score(y_test, dengeli_tahmin),
        f1_score(y_test, dengeli_tahmin),
        roc_auc_score(y_test, dengeli_olasilik)
    ]
})
karsilastirma

In [ ]:
karsilastirma.set_index('Metrik').plot(kind='bar', figsize=(10, 6), color=['#3498db', '#e67e22'])
plt.title('Temel Model ve Dengeli Model Karşılaştırması')
plt.ylabel('Değer')
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

In [ ]:
cm_dengeli = confusion_matrix(y_test, dengeli_tahmin)

plt.figure(figsize=(6, 5))
sns.heatmap(cm_dengeli, annot=True, fmt='d', cmap='Oranges', xticklabels=['Kabul Etmedi', 'Kabul Etti'], yticklabels=['Kabul Etmedi', 'Kabul Etti'])
plt.xlabel('Tahmin Edilen')
plt.ylabel('Gerçek')
plt.title('Dengeli Model Karmaşıklık Matrisi')
plt.tight_layout()
plt.show()

## Katsayı Yorumlaması

Lojistik regresyon katsayıları, üstel dönüşüm sonrası odds oranı (olasılık oranı) olarak yorumlanmıştır. 1'den büyük bir odds oranı ilgili değişkenin artmasının kampanyayı kabul etme olasılığını artırdığını gösterir.

In [ ]:
katsayi_df = pd.DataFrame({
    'Değişken': ozellikler,
    'Katsayı': dengeli_model.coef_[0],
    'Odds Oranı': np.exp(dengeli_model.coef_[0])
}).sort_values('Odds Oranı', ascending=False)

katsayi_df

In [ ]:
plt.figure(figsize=(9, 6))
sns.barplot(data=katsayi_df, x='Odds Oranı', y='Değişken', palette='viridis')
plt.axvline(x=1, color='red', linestyle='--', label='Etkisiz Sınır (Odds=1)')
plt.title('Değişkenlere Göre Odds Oranları')
plt.legend()
plt.tight_layout()
plt.show()

## Eşik Değeri Optimizasyonu

Varsayılan 0.5 eşik değeri yerine, farklı eşik değerlerinde kesinlik, duyarlılık ve F1 skorunun nasıl değiştiği incelenerek en dengeli eşik değeri belirlenmiştir.

In [ ]:
esik_araligi = np.arange(0.1, 0.9, 0.02)
kesinlik_listesi = []
duyarlilik_listesi = []
f1_listesi = []

for esik in esik_araligi:
    tahmin_esikli = (dengeli_olasilik >= esik).astype(int)
    kesinlik_listesi.append(precision_score(y_test, tahmin_esikli, zero_division=0))
    duyarlilik_listesi.append(recall_score(y_test, tahmin_esikli, zero_division=0))
    f1_listesi.append(f1_score(y_test, tahmin_esikli, zero_division=0))

plt.figure(figsize=(10, 6))
plt.plot(esik_araligi, kesinlik_listesi, label='Kesinlik', color='#2980b9')
plt.plot(esik_araligi, duyarlilik_listesi, label='Duyarlılık', color='#27ae60')
plt.plot(esik_araligi, f1_listesi, label='F1 Skoru', color='#c0392b')
plt.xlabel('Karar Eşiği')
plt.ylabel('Skor')
plt.title('Eşik Değerine Göre Kesinlik, Duyarlılık ve F1 Skoru')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
en_iyi_esik = esik_araligi[int(np.argmax(f1_listesi))]
print(f'En yüksek F1 skorunu veren eşik değeri: {en_iyi_esik:.2f}')
print(f'Bu eşikteki F1 skoru: {max(f1_listesi):.4f}')

## Çapraz Doğrulama

Modelin farklı veri bölünmelerinde de tutarlı performans gösterip göstermediğini kontrol etmek için 5 katlı tabakalı çapraz doğrulama uygulanmıştır.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_auc_skorlari = cross_val_score(
    LogisticRegression(max_iter=500, class_weight='balanced', random_state=42),
    X, y, cv=cv, scoring='roc_auc'
)

plt.figure(figsize=(8, 5))
plt.bar(range(1, 6), cv_auc_skorlari, color='#8e44ad')
plt.axhline(y=cv_auc_skorlari.mean(), color='red', linestyle='--', label=f'Ortalama ROC-AUC = {cv_auc_skorlari.mean():.3f}')
plt.xlabel('Kat Numarası')
plt.ylabel('ROC-AUC Skoru')
plt.title('5 Katlı Çapraz Doğrulama ROC-AUC Skorları')
plt.xticks(range(1, 6))
plt.legend()
plt.tight_layout()
plt.show()

## Hiperparametre Optimizasyonu

Regülarizasyon gücünü kontrol eden C parametresi ve ceza türü (penalty) için ızgara arama (grid search) uygulanarak en iyi kombinasyon bulunmuştur.

In [ ]:
parametre_izgarasi = {
    'C': [0.01, 0.1, 1, 10, 100],
    'penalty': ['l1', 'l2'],
    'solver': ['liblinear']
}

izgara_arama = GridSearchCV(
    LogisticRegression(max_iter=500, class_weight='balanced', random_state=42),
    param_grid=parametre_izgarasi,
    cv=cv,
    scoring='roc_auc'
)
izgara_arama.fit(X_train_scaled, y_train)

print(f'En iyi parametreler: {izgara_arama.best_params_}')
print(f'En iyi çapraz doğrulama ROC-AUC skoru: {izgara_arama.best_score_:.4f}')

In [ ]:
en_iyi_model = izgara_arama.best_estimator_
en_iyi_tahmin = en_iyi_model.predict(X_test_scaled)
en_iyi_olasilik = en_iyi_model.predict_proba(X_test_scaled)[:, 1]

son_metrikler = pd.DataFrame({
    'Metrik': ['Doğruluk', 'Kesinlik', 'Duyarlılık', 'F1 Skoru', 'ROC-AUC'],
    'Değer': [
        accuracy_score(y_test, en_iyi_tahmin),
        precision_score(y_test, en_iyi_tahmin),
        recall_score(y_test, en_iyi_tahmin),
        f1_score(y_test, en_iyi_tahmin),
        roc_auc_score(y_test, en_iyi_olasilik)
    ]
})
son_metrikler

In [ ]:
fpr_son, tpr_son, _ = roc_curve(y_test, en_iyi_olasilik)
roc_auc_son = roc_auc_score(y_test, en_iyi_olasilik)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='#95a5a6', linewidth=2, linestyle='--', label=f'Dengeli Model (AUC = {roc_auc:.3f})')
plt.plot(fpr_son, tpr_son, color='#2980b9', linewidth=2, label=f'Optimize Edilmiş Model (AUC = {roc_auc_son:.3f})')
plt.plot([0, 1], [0, 1], 'k--', linewidth=1)
plt.xlabel('Yanlış Pozitif Oranı')
plt.ylabel('Doğru Pozitif Oranı')
plt.title('Dengeli Model ve Optimize Edilmiş Model ROC Karşılaştırması')
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

## Sonuç

Temel lojistik regresyon modeli yüksek doğruluk gösterse de sınıf dengesizliği nedeniyle kampanyayı kabul eden müşterilerin büyük bir kısmını kaçırmaktadır. Sınıf ağırlıklandırması uygulanan model, duyarlılığı belirgin şekilde artırarak azınlık sınıfını daha iyi yakalamaktadır. Katsayı analizi; toplam harcama, gelir ve son alışverişten bu yana geçen süre gibi değişkenlerin kampanya yanıtını en çok etkileyen faktörler olduğunu göstermektedir. Hiperparametre optimizasyonu sonrasında elde edilen model, çapraz doğrulama ile de doğrulanan istikrarlı bir ROC-AUC performansı sunmaktadır.